# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and processing the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Install mlcroissant if not yet installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the FAIR<sup>2</sup> dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {getattr(metadata, 'name', '')}\nDescription: {getattr(metadata, 'description', '')}")

## 2. Data Overview

List all available record sets, their field `@id`s, and the corresponding columns. Entities are referenced by their `@id` in accordance with the Croissant schema.

In [ ]:
from pprint import pprint

# List all record sets and their field/column ids
record_sets_metadata = getattr(metadata, "record_sets", [])
print("Available record sets and their fields/columns by @id:")
record_set_ids = []
for recset in record_sets_metadata:
    recset_id = getattr(recset, '@id', None) or getattr(recset, 'id', None)
    record_set_ids.append(recset_id)
    recset_name = getattr(recset, 'name', '')
    print(f"\nRecordSet: {recset_name} (@id: {recset_id})")
    fields = getattr(recset, 'fields', [])
    if fields:
        print("  Fields:")
        for field in fields:
            print(f"    - {getattr(field, 'name', '')} (@id: {getattr(field, '@id', getattr(field, 'id', '?'))})")
            columns = getattr(field, "columns", [])
            for col in columns:
                print(f"      Column: {getattr(col, 'name', '')} (@id: {getattr(col, '@id', getattr(col, 'id', '?'))})")

## 3. Data Extraction

Load records from a selected record set. (Below, we choose the principal protein record set and show its fields.)

> **Note**: To reference entities, use their `@id` as shown in the previous cell output.

In [ ]:
# List all record_set @id's you want to load (example chosen based on data overview above)
# If unsure, check the printed @id from the overview cell above.
# Here we use the main record set '@id', e.g., 'cr:main' (update to the actual @id in your dataset)

record_sets_to_load = record_set_ids  # use all available record sets
dataframes = {}

for recset_id in record_sets_to_load:
    records = list(dataset.records(record_set=recset_id))
    if records:
        dataframes[recset_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[recset_id])} records for RecordSet {recset_id}")
    else:
        print(f"No records found for RecordSet {recset_id}")

# Show columns from the first loaded record set
example_recset_id = next(iter(dataframes.keys()))
print(f"Available fields/columns in '{example_recset_id}':\n", dataframes[example_recset_id].columns.tolist())
dataframes[example_recset_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filtering, normalization, and grouping. The following example uses a numeric field and a group field from the record set columns by their `@id`.

Please update `<numeric_field_id>` and `<group_field_id>` to actual field `@id`s appropriate for your analysis as needed.

In [ ]:
# Select the first (e.g. main protein) record set for EDA
df = dataframes[example_recset_id]

# Choose a numeric field and a group-field based on column names (as shown previously)
# For instance, common in proteomics: 'cr:field_mw' (molecular weight), or abundance, etc.
# Update these identifiers to fit the printed column names, as needed.
numeric_field_id = None
group_field_id = None

# Let's auto-guess a numeric field and a grouping field from the DataFrame
for col in df.columns:
    # Try to select numeric field
    if numeric_field_id is None and pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
    # Pick something that may be suitable for grouping, e.g. protein name/type
    if group_field_id is None and (df[col].dtype == object and 'name' in col.lower() or 'type' in col.lower() or 'group' in col.lower()):
        group_field_id = col
    if numeric_field_id and group_field_id:
        break

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

# Filtering example: keep only rows where the numeric field is > threshold
if numeric_field_id and numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

    # Grouping example
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print('No suitable numeric field found for EDA. Please check your schema and update the field id accordingly.')

## 5. Visualization

Visualize the distribution of the chosen numeric field and its relationship with the group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset via its FAIR<sup>2</sup> Croissant schema with `mlcroissant`. We reviewed its record sets, field, and column structure (referenced uniquely by `@id`) and performed basic exploratory data analysis on the main record set. This demonstrates how to flexibly process any Croissant-compatible dataset for reproducible data science workflows.

You can now extend this notebook by referencing addition fields, record sets, or filtering logic using entity `@id`s as needed for your research.